### SpendDNA - Financial Transaction Analysis

### Name : Mayuresh Khare
### Project : SpendDNA (Minor Project 2)
### Academy : The Unlox Academy
### Tools Used : Python, NumPy, Pandas
### Dataset : Rahul Sharma Transactions (Jan–Jun 2024)

### Objective: The objective of this project is to analyze six months of bank transaction data by cleaning raw financial records, extracting vendors, categorizing expenses, detecting anomalies, identifying spending patterns, and generating a detailed financial report using only Python, NumPy, and Pandas.

In [103]:
import pandas as pd
import numpy as np
from datetime import datetime

df = pd.read_csv("bank_transactions.csv")

In [104]:
print("Shape of Dataset :", df.shape)

print("\nColumns :")

print(df.columns.tolist())

Shape of Dataset : (1328, 8)

Columns :
['Date', 'Time', 'Description', 'Type', 'Amount', 'Balance', 'Mode', 'Ref']


In [105]:
df.head()

,Date,Time,Description,Type,Amount,Balance,Mode,Ref
0,2024-01-01,03:11,AMAZON SELLER SVCS,Debit,₹2462,678275.0,UPI,TXN190872
1,01-Jan-24,05:44,BHIM-BMTC,DR,50.00,681007.0,UPI,TXN143064
2,01-Jan-24,09:35,NEFT-TECHCRUSH LABS-SALARY MAY24,CR,₹84728,484728.0,NEFT,TXN246316
3,2024-01-01,14:07,UPI-AMAN-8934@OKAXIS,Debit,₹1828,-748745.0,UPI,TXN569226
4,01 Jan 2024,14:23,BHIM-BLINKIT,Debit,270.00,680737.0,UPI,TXN968962


In [106]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1328 entries, 0 to 1327
Data columns (total 8 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   Date         1328 non-null   object 
 1   Time         1328 non-null   object 
 2   Description  1328 non-null   object 
 3   Type         1328 non-null   object 
 4   Amount       1328 non-null   object 
 5   Balance      1328 non-null   float64
 6   Mode         1328 non-null   object 
 7   Ref          1328 non-null   object 
dtypes: float64(1), object(7)
memory usage: 83.1+ KB


In [107]:
df.isnull().sum()

,0
Date,0
Time,0
Description,0
Type,0
Amount,0
Balance,0
Mode,0
Ref,0


# Feature 1- Data Cleaning

In [108]:
# Converting Date column into datetime format
def parse_date(date):

    if pd.isna(date):
        return pd.NaT

    formats = [
        "%d/%m/%y",
        "%Y-%m-%d",
        "%d-%b-%y",
        "%d %b %Y"
    ]

    for fmt in formats:
        try:
            return datetime.strptime(date, fmt)
        except ValueError:
            continue

    return pd.NaT


df["Date"] = df["Date"].apply(parse_date)

print("Invalid Dates:", df["Date"].isna().sum())

Invalid Dates: 0


In [109]:
print(df["Date"].head(10))

0   2024-01-01
1   2024-01-01
2   2024-01-01
3   2024-01-01
4   2024-01-01
5   2024-01-01
6   2024-01-01
7   2024-01-01
8   2024-01-02
9   2024-01-02
Name: Date, dtype: datetime64[ns]


In [110]:
print(df["Date"].min())
print(df["Date"].max())

2024-01-01 00:00:00
2024-06-30 00:00:00


In [111]:
# Cleaning Amount column

df["Amount"] = (
    df["Amount"]
    .astype(str)
    .str.replace("₹", "", regex=False)
    .str.replace("Rs.", "", regex=False)
    .str.replace(",", "", regex=False)
    .str.strip()
)

df["Amount"] = pd.to_numeric(
    df["Amount"],
    errors="coerce"
)

print("Amount column cleaned successfully.\n")

print(df["Amount"].head())

print("\nData Type :", df["Amount"].dtype)

Amount column cleaned successfully.

0     2462.0
1       50.0
2    84728.0
3     1828.0
4      270.0
Name: Amount, dtype: float64

Data Type : float64


In [112]:
# Standardizing transaction type

df["Type"] = (
    df["Type"]
    .str.lower()
    .replace({
        "dr": "debit",
        "cr": "credit"
    })
)

print(df["Type"].value_counts())

Type
debit     1322
credit       6
Name: count, dtype: int64


In [113]:
# Checking duplicates

duplicates = df.duplicated().sum()

print("Duplicate Rows :", duplicates)

# Removing duplicates

df = df.drop_duplicates()

print("\nShape After Removing Duplicates :", df.shape)

Duplicate Rows : 18

Shape After Removing Duplicates : (1310, 8)


In [114]:
print("=" * 50)

print("TRANSACTION PARSER SUMMARY")

print("=" * 50)

print(f"Total Transactions : {len(df)}")

print(f"Duplicate Rows Removed : {duplicates}")

print(f"Invalid Dates : {df['Date'].isna().sum()}")

print(f"Invalid Amounts : {df['Amount'].isna().sum()}")

print("\nData Types\n")

print(df.dtypes)

TRANSACTION PARSER SUMMARY
Total Transactions : 1310
Duplicate Rows Removed : 18
Invalid Dates : 0
Invalid Amounts : 0

Data Types

Date           datetime64[ns]
Time                   object
Description            object
Type                   object
Amount                float64
Balance               float64
Mode                   object
Ref                    object
dtype: object


In [115]:
print(df["Date"].head(20))

0    2024-01-01
1    2024-01-01
2    2024-01-01
3    2024-01-01
4    2024-01-01
5    2024-01-01
6    2024-01-01
7    2024-01-01
8    2024-01-02
9    2024-01-02
10   2024-01-02
11   2024-01-02
12   2024-01-02
13   2024-01-02
14   2024-01-02
15   2024-01-02
16   2024-01-02
17   2024-01-02
18   2024-01-03
19   2024-01-03
Name: Date, dtype: datetime64[ns]


# Feature 2- Vendor Extraction

In [116]:
print("Total Unique Descriptions :", df["Description"].nunique())

print("\nFirst 50 Unique Descriptions:\n")

for desc in df["Description"].unique()[:50]:
    print(desc)

Total Unique Descriptions : 283

First 50 Unique Descriptions:

AMAZON SELLER SVCS
BHIM-BMTC
NEFT-TECHCRUSH LABS-SALARY MAY24
UPI-AMAN-8934@OKAXIS
BHIM-BLINKIT
BHIM ZEPTO
UPI-UBER-2426@HDFCBANK
POS SWIGGY BANGALORE
UPI-GROWWPAY@HDFCBANK
OLA ELECTRIC
BMS MOVIE TICKETS
POS OLA-PRIME
SWIGGY-INSTAMART
UPI-STARBUCKS@AXIS
UPI-THIRDWAVE@OKAXIS
ANI Technologies
BMTC BUS PASS
POS TRUFFLES
FLIPKART INDIA
POS SWIGGY-RESTAURANT
GROFERS INDIA P L
POS UBER BANGALORE
BANGALORE ELEC SUPPLY
TWC INDIA
UPI-BESCOM-BILL@HDFCBANK
UPI-AMAN-0816@OKAXIS
ROPPEN TRANSPORTATION
OLA CABS
POS ZOMATO
UPI-AMAZONPAY@HDFCBANK
POS BLINKIT
IMPS-RENT-LANDLORD-75500265
ZOMATO MEDIA P L
UPI-ANKIT-6430@OKAXIS
UPI-OLACABS@HDFCBANK
UPI-JIORECHARGE@PAYTM
UPI-CCD@HDFCBANK
Swiggy*Order
INSTAMART BANGALORE
UPI-ZOMATO-LIMITED@PAYTM
AVENUE SUPERMARTS
POS HP PETROL STATION
UPI-VIKAS-6060@OKAXIS
POS BANGALORE RESTAURANT
UPI-ZERODHA-COIN@AXIS
BHIM SWIGGY
UPI-BOOKMYSHOW@HDFCBANK
BLINKIT BANGALORE
POS ZEPTO BANGALORE
UPI-DMART@OKAXIS


In [117]:
# Vendor Mapping Dictionary

vendor_dict = {
    "Swiggy": ["SWIGGY", "BUNDL", "INSTAMART"],
    "Zomato": ["ZOMATO"],
    "Blinkit": ["BLINKIT"],
    "Zepto": ["ZEPTO"],
    "Amazon": ["AMAZON", "AMZN"],
    "Flipkart": ["FLIPKART"],
    "Groww": ["GROWW"],
    "Zerodha": ["ZERODHA"],
    "Uber": ["UBER"],
    "Ola": ["OLA", "ANI", "ROPPEN"],
    "Starbucks": ["STARBUCKS"],
    "Third Wave Coffee": ["THIRDWAVE", "TWC"],
    "Cafe Coffee Day": ["CCD"],
    "Truffles": ["TRUFFLES"],
    "BookMyShow": ["BOOKMYSHOW", "BMS"],
    "DMart": ["DMART", "AVENUE SUPERMARTS"],
    "Grofers": ["GROFERS"],
    "HP Petrol": ["HP PETROL"],
    "BESCOM": ["BESCOM", "BANGALORE ELEC"],
    "Jio": ["JIO"],
    "BMTC": ["BMTC"]
}

In [118]:
print("Total Vendors in Dictionary :", len(vendor_dict))

print("\nVendor Names:\n")

for vendor in vendor_dict:
    print(vendor)

Total Vendors in Dictionary : 21

Vendor Names:

Swiggy
Zomato
Blinkit
Zepto
Amazon
Flipkart
Groww
Zerodha
Uber
Ola
Starbucks
Third Wave Coffee
Cafe Coffee Day
Truffles
BookMyShow
DMart
Grofers
HP Petrol
BESCOM
Jio
BMTC


In [119]:
# Function to extract vendor name

def extract_vendor(description):

    # Convert description to uppercase
    description = str(description).upper()

    # Check each vendor
    for vendor, keywords in vendor_dict.items():

        for keyword in keywords:

            if keyword in description:
                return vendor

    # Check for Person-to-Person transfers
    if "UPI-" in description and "@OK" in description:
      return "P2P Transfer"

    return "Uncategorized"

In [120]:
print(extract_vendor("POS SWIGGY BANGALORE"))
print(extract_vendor("UPI-AMAZONPAY@HDFCBANK"))
print(extract_vendor("UPI-GROWWPAY@HDFCBANK"))
print(extract_vendor("UPI-AMAN-8934@OKAXIS"))
print(extract_vendor("POS ZEPTO BANGALORE"))

Swiggy
Amazon
Groww
P2P Transfer
Zepto


In [121]:
# Apply extract_vendor() to every transaction

df["Vendor"] = df["Description"].apply(extract_vendor)

print("Vendor extraction completed successfully!")

Vendor extraction completed successfully!


In [122]:
print(df[["Description", "Vendor"]].head(15))

                         Description         Vendor
0                 AMAZON SELLER SVCS         Amazon
1                          BHIM-BMTC           BMTC
2   NEFT-TECHCRUSH LABS-SALARY MAY24  Uncategorized
3               UPI-AMAN-8934@OKAXIS   P2P Transfer
4                       BHIM-BLINKIT        Blinkit
5                         BHIM ZEPTO          Zepto
6             UPI-UBER-2426@HDFCBANK           Uber
7                       BHIM-BLINKIT        Blinkit
8               POS SWIGGY BANGALORE         Swiggy
9              UPI-GROWWPAY@HDFCBANK          Groww
10                      OLA ELECTRIC            Ola
11                 BMS MOVIE TICKETS     BookMyShow
12                     POS OLA-PRIME            Ola
13                  SWIGGY-INSTAMART         Swiggy
14                UPI-STARBUCKS@AXIS      Starbucks


In [123]:
print(df["Vendor"].value_counts())

Vendor
Uncategorized        287
Swiggy               243
Zomato               121
Ola                  101
Amazon                86
Uber                  71
Zepto                 58
Flipkart              43
Starbucks             42
P2P Transfer          42
Blinkit               40
BMTC                  37
DMart                 22
Third Wave Coffee     20
BESCOM                15
Grofers               15
Zerodha               14
BookMyShow            11
Groww                  9
Truffles               9
Cafe Coffee Day        9
HP Petrol              9
Jio                    6
Name: count, dtype: int64


In [124]:
# Display unique uncategorized descriptions

uncategorized = df[df["Vendor"] == "Uncategorized"]["Description"].unique()

print("Total Uncategorized Descriptions:", len(uncategorized))
print()

for desc in uncategorized:
    print(desc)

Total Uncategorized Descriptions: 87

NEFT-TECHCRUSH LABS-SALARY MAY24
IMPS-RENT-LANDLORD-75500265
POS BANGALORE RESTAURANT
POS CAFE COFFEE DAY
UPI-MYNTRA@HDFCBANK
UPI-BWSSB@AXIS
UPI-NETFLIX-MEMB@HDFCBANK
AIRTEL POSTPAID
POS NYKAA BANGALORE
VI POSTPAID
POS BPCL PETROL PUMP
POS INDIAN OIL BANGALORE
UPI-SPOTIFY-IN@AXIS
POS DINEOUT
COFFEE DAY GLOBAL
UPI-NYKAA@AXIS
RAPIDO BIKE TAXI
DISNEY HOTSTAR
POS EMPIRE RESTAURANT
ATM-WDL-SBI-5715
POS MEGHANA FOODS
UPI-IOC9075@HDFCBANK
FKART INTRNET
UPI-BIGBASKET@HDFCBANK
MYNTRA-ORDER
MYNTRA DESIGNS
POS MYNTRA
BIGBASKET BANGALORE
POS THIRD WAVE COFFEE
SPOTIFY INDIA
IMPS-RENT-LANDLORD-39598076
UPI-RESTAURANT-7285@PAYTM
NETFLIX ENTERTAINMENT
FSN E-COMMERCE
UPI-RESTAURANT-7678@PAYTM
ATM-WDL-HDFC-4942
BIGTREE ENTERTAINMENT
INNOVATIVE RETAIL
UPI-RESTAURANT-3715@PAYTM
STAR INDIA PVT LTD
UPI-RESTAURANT-0251@PAYTM
KIRANAKART TECH
IMPS-RENT-LANDLORD-51148088
UPI-RESTAURANT-6377@PAYTM
UPI-HOTSTAR@AXIS
ATM-WDL-HDFC-9140
ATM-WDL-SBI-4080
NETFLIX.COM
UPI-IOC0494@HD

In [125]:
# Add these new vendors to the existing dictionary
vendor_dict.update({
    "Salary": ["SALARY"],
    "Rent": ["RENT", "LANDLORD"],
    "Restaurant": ["RESTAURANT", "EMPIRE", "MEGHANA", "DINEOUT"],
    "Cafe Coffee Day": ["CCD", "COFFEE DAY"],
    "Myntra": ["MYNTRA", "FSN"],
    "Nykaa": ["NYKAA"],
    "Netflix": ["NETFLIX"],
    "Spotify": ["SPOTIFY"],
    "Disney Hotstar": ["HOTSTAR", "STAR INDIA"],
    "BigBasket": ["BIGBASKET", "KIRANAKART"],
    "Airtel": ["AIRTEL", "BHARTI AIRTEL"],
    "Vi": ["VI", "VODAFONE IDEA"],
    "Rapido": ["RAPIDO"],
    "Indian Oil": ["IOC", "INDIAN OIL"],
    "BPCL": ["BPCL"],
    "ATM Withdrawal": ["ATM-WDL"],
    "BWSSB": ["BWSSB"],
    "BookMyShow": ["BOOKMYSHOW", "BMS", "BIGTREE"],
    "Flipkart": ["FLIPKART", "FKART"],
    "DMart": ["DMART", "AVENUE SUPERMARTS", "INNOVATIVE RETAIL"]
})

In [126]:
# Recreate Vendor column using updated dictionary

df["Vendor"] = df["Description"].apply(extract_vendor)

print("Vendor column updated successfully!")

Vendor column updated successfully!


In [127]:
print(df["Vendor"].value_counts())

Vendor
Swiggy               243
Zomato               121
Ola                  101
Amazon                86
Uber                  71
Restaurant            64
Zepto                 58
Flipkart              47
Starbucks             42
Rapido                41
Blinkit               40
BMTC                  37
DMart                 30
Cafe Coffee Day       26
BigBasket             24
Myntra                23
Third Wave Coffee     20
ATM Withdrawal        17
Indian Oil            17
Nykaa                 16
BESCOM                15
Grofers               15
P2P Transfer          14
Zerodha               14
Disney Hotstar        13
BookMyShow            13
Vi                    12
Uncategorized         11
Netflix               10
HP Petrol              9
Groww                  9
Truffles               9
Spotify                8
BWSSB                  8
Salary                 6
Jio                    6
Rent                   6
Airtel                 6
BPCL                   2
Name: count, dtype

# Feature 3- Category Tagging

In [128]:
# Category Mapping Dictionary
category_dict = {
    "Swiggy": "Food Delivery",
    "Zomato": "Food Delivery",
    "Restaurant": "Food Delivery",
    "Blinkit": "Quick Commerce",
    "Zepto": "Quick Commerce",
    "BigBasket": "Quick Commerce",
    "Grofers": "Quick Commerce",
    "Amazon": "E-Commerce",
    "Flipkart": "E-Commerce",
    "Myntra": "E-Commerce",
    "Nykaa": "E-Commerce",
    "Uber": "Transport",
    "Ola": "Transport",
    "Rapido": "Transport",
    "BMTC": "Transport",
    "Groww": "Investment",
    "Zerodha": "Investment",
    "Salary": "Income",
    "Rent": "Housing",
    "BESCOM": "Utilities",
    "BWSSB": "Utilities",
    "Jio": "Utilities",
    "Airtel": "Utilities",
    "Vi": "Utilities",
    "Netflix": "Entertainment",
    "Spotify": "Entertainment",
    "Disney Hotstar": "Entertainment",
    "BookMyShow": "Entertainment",
    "Starbucks": "Cafe",
    "Cafe Coffee Day": "Cafe",
    "Third Wave Coffee": "Cafe",
    "Indian Oil": "Fuel",
    "HP Petrol": "Fuel",
    "BPCL": "Fuel",
    "ATM Withdrawal": "Cash Withdrawal",
    "P2P Transfer": "Personal Transfer",
    "DMart": "Groceries",
    "Truffles": "Restaurant",
    "Uncategorized": "Others"
}

In [129]:
# Create Category column

df["Category"] = df["Vendor"].map(category_dict)

print("Category column created successfully!")

Category column created successfully!


In [130]:
print(df[["Vendor", "Category"]].head(20))

               Vendor           Category
0              Amazon         E-Commerce
1                BMTC          Transport
2              Salary             Income
3        P2P Transfer  Personal Transfer
4             Blinkit     Quick Commerce
5               Zepto     Quick Commerce
6                Uber          Transport
7             Blinkit     Quick Commerce
8              Swiggy      Food Delivery
9               Groww         Investment
10                Ola          Transport
11         BookMyShow      Entertainment
12                Ola          Transport
13             Swiggy      Food Delivery
14          Starbucks               Cafe
15  Third Wave Coffee               Cafe
16                Ola          Transport
17               BMTC          Transport
18           Truffles         Restaurant
19           Flipkart         E-Commerce


In [131]:
print(df["Category"].value_counts())

Category
Food Delivery        428
Transport            250
E-Commerce           172
Quick Commerce       137
Cafe                  88
Utilities             47
Entertainment         44
Groceries             30
Fuel                  28
Investment            23
Cash Withdrawal       17
Personal Transfer     14
Others                11
Restaurant             9
Income                 6
Housing                6
Name: count, dtype: int64


# Feature 4- Spending Overview

In [132]:
# Separate Income and Expenses

income = df[df["Category"] == "Income"]

# Expenses excluding non-consumption transactions

expenses = df[
    (df["Type"] == "debit") &
    (~df["Category"].isin([
        "Investment",
        "Personal Transfer",
        "Cash Withdrawal"
    ]))
]

print("Income Transactions :", len(income))
print("Expense Transactions :", len(expenses))

Income Transactions : 6
Expense Transactions : 1250


In [133]:
# Calculate totals
total_income = income["Amount"].sum()

total_expense = expenses["Amount"].sum()

net_savings = total_income - total_expense

print(f"Total Income  : ₹{total_income:,.2f}")
print(f"Total Expense : ₹{total_expense:,.2f}")
print(f"Net Savings   : ₹{net_savings:,.2f}")

Total Income  : ₹509,774.00
Total Expense : ₹1,365,402.00
Net Savings   : ₹-855,628.00


In [134]:
# Savings Rate

savings_rate = (net_savings / total_income) * 100

print(f"Savings Rate : {savings_rate:.2f}%")

Savings Rate : -167.84%


In [135]:
# Top Spending Categories

category_spend = (
    expenses
    .groupby("Category")["Amount"]
    .sum()
    .sort_values(ascending=False)
)

print(category_spend)

Category
E-Commerce        603877.0
Food Delivery     260731.0
Housing           108000.0
Fuel               89303.0
Quick Commerce     77799.0
Transport          57474.0
Utilities          46674.0
Groceries          46662.0
Cafe               27897.0
Entertainment      26764.0
Restaurant         16673.0
Others              3548.0
Name: Amount, dtype: float64


In [136]:
# Top Spending Vendors

vendor_spend = (
    expenses
    .groupby("Vendor")["Amount"]
    .sum()
    .sort_values(ascending=False)
)

print(vendor_spend.head(10))

Vendor
Amazon        328530.0
Flipkart      177510.0
Rent          108000.0
Swiggy        104351.0
Restaurant    101064.0
Myntra         73825.0
Indian Oil     56335.0
Zomato         55316.0
DMart          46662.0
Ola            28372.0
Name: Amount, dtype: float64


# Feature 5– Monthly Trends

In [137]:
# Create Month column

df["Month"] = df["Date"].dt.strftime("%b")

print(df[["Date", "Month"]].head())

        Date Month
0 2024-01-01   Jan
1 2024-01-01   Jan
2 2024-01-01   Jan
3 2024-01-01   Jan
4 2024-01-01   Jan


In [138]:
# Recreate Income and Expense DataFrames after adding Month column

income = df[df["Category"] == "Income"]

expenses = df[
    (df["Type"] == "debit") &
    (~df["Category"].isin([
        "Investment",
        "Personal Transfer",
        "Cash Withdrawal"
    ]))
]

In [139]:
print(expenses.columns)

Index(['Date', 'Time', 'Description', 'Type', 'Amount', 'Balance', 'Mode',
       'Ref', 'Vendor', 'Category', 'Month'],
      dtype='object')


In [140]:
# Recreate filtered DataFrames

income = df[df["Category"] == "Income"].copy()

expenses = df[
    (df["Type"] == "debit") &
    (~df["Category"].isin([
        "Investment",
        "Personal Transfer",
        "Cash Withdrawal"
    ]))
].copy()

# Create Month column inside both DataFrames
income["Month"] = income["Date"].dt.strftime("%b")
expenses["Month"] = expenses["Date"].dt.strftime("%b")

# Monthly Spending
monthly_spend = (
    expenses.groupby("Month")["Amount"]
    .sum()
    .reindex(["Jan", "Feb", "Mar", "Apr", "May", "Jun"])
)

print(monthly_spend)

Month
Jan    242812.0
Feb    209786.0
Mar    251399.0
Apr    191093.0
May    217266.0
Jun    253046.0
Name: Amount, dtype: float64


In [141]:
# Monthly Income

monthly_income = (
    income.groupby("Month")["Amount"]
    .sum()
    .reindex(["Jan", "Feb", "Mar", "Apr", "May", "Jun"])
)

print(monthly_income)

Month
Jan    84728.0
Feb    84724.0
Mar    84701.0
Apr    84736.0
May    85393.0
Jun    85492.0
Name: Amount, dtype: float64


In [142]:
# Monthly Summary

monthly_summary = pd.DataFrame({
    "Income": monthly_income,
    "Expense": monthly_spend
})

monthly_summary["Savings"] = (
    monthly_summary["Income"] - monthly_summary["Expense"]
)

print(monthly_summary)

        Income   Expense   Savings
Month                             
Jan    84728.0  242812.0 -158084.0
Feb    84724.0  209786.0 -125062.0
Mar    84701.0  251399.0 -166698.0
Apr    84736.0  191093.0 -106357.0
May    85393.0  217266.0 -131873.0
Jun    85492.0  253046.0 -167554.0


In [143]:
highest_month = monthly_spend.idxmax()
highest_amount = monthly_spend.max()

print(f"Highest Spending Month : {highest_month}")
print(f"Amount Spent : ₹{highest_amount:,.2f}")

Highest Spending Month : Jun
Amount Spent : ₹253,046.00


# Feature 6- Time Analysis

In [144]:
def get_time_period(time):

    hour = int(str(time).split(":")[0])

    if 5 <= hour < 12:
        return "Morning"

    elif 12 <= hour < 17:
        return "Afternoon"

    elif 17 <= hour < 21:
        return "Evening"

    else:
        return "Night"


df["Time Period"] = df["Time"].apply(get_time_period)

print("Time Period column created successfully!")

Time Period column created successfully!


In [145]:
print(df[["Time","Time Period"]].head(10))

    Time Time Period
0  03:11       Night
1  05:44     Morning
2  09:35     Morning
3  14:07   Afternoon
4  14:23   Afternoon
5  14:48   Afternoon
6  14:50   Afternoon
7  21:44       Night
8  05:18     Morning
9  06:55     Morning


In [146]:
time_count = df["Time Period"].value_counts()

print(time_count)

Time Period
Morning      354
Evening      340
Afternoon    314
Night        302
Name: count, dtype: int64


In [147]:
expenses = df[
    (df["Type"] == "debit") &
    (~df["Category"].isin([
        "Investment",
        "Personal Transfer",
        "Cash Withdrawal"
    ]))
].copy()

In [148]:
time_spend = (
    expenses
    .groupby("Time Period")["Amount"]
    .sum()
)

print(time_spend)

Time Period
Afternoon    433417.0
Evening      316214.0
Morning      333255.0
Night        282516.0
Name: Amount, dtype: float64


In [149]:
peak_time = time_spend.idxmax()

peak_amount = time_spend.max()

print(f"Peak Spending Time : {peak_time}")

print(f"Amount Spent : ₹{peak_amount:,.2f}")

Peak Spending Time : Afternoon
Amount Spent : ₹433,417.00


# Feature 7– Anomaly Detection

In [150]:
# Large Expense Transactions (> ₹20,000)

anomalies = df[
    (df["Type"] == "debit") &
    (df["Amount"] > 20000)
]

print("Total Large Expense Transactions :", len(anomalies))

Total Large Expense Transactions : 2


In [151]:
print(
    anomalies[
        ["Date",
         "Description",
         "Vendor",
         "Category",
         "Amount"]
    ].sort_values(
        by="Amount",
        ascending=False
    )
)

           Date             Description  Vendor    Category   Amount
1298 2024-06-26    AMAZONIN MARKETPLACE  Amazon  E-Commerce  22008.0
269  2024-02-07  UPI-AMAZONPAY@HDFCBANK  Amazon  E-Commerce  21986.0


In [152]:
print("Top 10 Highest Transactions\n")

print(
    anomalies.nlargest(
        10,
        "Amount"
    )[
        [
            "Date",
            "Vendor",
            "Category",
            "Amount"
        ]
    ]
)

Top 10 Highest Transactions

           Date  Vendor    Category   Amount
1298 2024-06-26  Amazon  E-Commerce  22008.0
269  2024-02-07  Amazon  E-Commerce  21986.0


In [153]:
largest = anomalies.loc[
    anomalies["Amount"].idxmax()
]

print("\nLargest Transaction")

print(largest[
    [
        "Date",
        "Description",
        "Vendor",
        "Amount"
    ]
])


Largest Transaction
Date            2024-06-26 00:00:00
Description    AMAZONIN MARKETPLACE
Vendor                       Amazon
Amount                      22008.0
Name: 1298, dtype: object


# Feature 8– Spending Archetypes

In [154]:
category_percentage = (
    expenses
    .groupby("Category")["Amount"]
    .sum()
    / expenses["Amount"].sum()
) * 100

print(category_percentage.round(2))

Category
Cafe               2.04
E-Commerce        44.23
Entertainment      1.96
Food Delivery     19.10
Fuel               6.54
Groceries          3.42
Housing            7.91
Others             0.26
Quick Commerce     5.70
Restaurant         1.22
Transport          4.21
Utilities          3.42
Name: Amount, dtype: float64


In [155]:
archetypes = []

if category_percentage.get("Food Delivery", 0) > 20:
    archetypes.append("🍔 Foodie")

if category_percentage.get("E-Commerce", 0) > 20:
    archetypes.append("🛍️ Shopaholic")

if category_percentage.get("Investment", 0) > 10:
    archetypes.append("📈 Investor")

if category_percentage.get("Entertainment", 0) > 10:
    archetypes.append("🎬 Entertainment Lover")

print("Spending Archetypes:")

for a in archetypes:
    print("-", a)

Spending Archetypes:
- 🛍️ Shopaholic


# Executive Summary

SpendDNA analyzes six months of bank transaction data to understand an individual's spending habits. The project cleans and processes raw transaction descriptions, identifies merchants, classifies expenses into meaningful categories, and generates business insights from the financial data.

The analysis includes vendor extraction, category mapping, monthly spending trends, transaction timing analysis, anomaly detection, and spending personality identification. These insights help users better understand where and how their money is being spent.

# Business Insights

After analyzing the transaction dataset, the following key insights were identified:

- The user receives a consistent monthly salary of approximately ₹85,000.
- Total expenses are significantly higher than the monthly income, resulting in negative savings.
- June recorded the highest monthly expenditure.
- Afternoon is the highest spending period of the day.
- Food Delivery and E-Commerce contribute the largest share of overall spending.
- Amazon is the highest spending vendor.
- High-value transactions are limited and mainly consist of salary credits and a few expensive purchases.
- Based on spending distribution, the user is classified as a **Shopaholic**, with E-Commerce contributing over 44% of total spending.

# Conclusion

SpendDNA successfully converts raw banking transaction data into meaningful financial insights using Python and Pandas.

The project demonstrates practical data cleaning, preprocessing, feature engineering, grouping, aggregation, and business analytics techniques. The generated insights provide a clear understanding of spending behavior and financial patterns.

This project highlights how data analytics can assist individuals in budgeting, identifying spending habits, and making informed financial decisions.

# Future Improvements

Future versions of SpendDNA can include:

- Interactive dashboard using Streamlit
- Data visualization using Matplotlib and Seaborn
- Automatic PDF report generation
- Machine Learning based spending prediction
- SMS and Email bank statement support
- AI-powered financial assistant